# Video Diffusion Models

Image diffusion with a time axis — which sounds like a small change and is not. Adding
frames turns a hard generative problem into a much harder one for two reasons that
dominate every design decision in the field:

1. **Cost explodes.** Full attention over a video is quadratic in `frames × height ×
   width`. Doubling the frame count quadruples the attention cost. Nothing else about
   the architecture matters if you cannot afford to run it.
2. **Temporal coherence is a new failure mode.** Every frame can be individually
   beautiful and the video still be unwatchable, because objects flicker, drift, change
   identity or violate physics between frames. No per-frame metric detects this.

Builds on [Diffusion Models](diffusion-models.ipynb),
[Diffusion Transformers](diffusion-transformer.ipynb), [U-Net](u-net.ipynb) and
[Attention Mechanisms](attention-mechanisms.ipynb). The practical library for running
these is [`diffusers`](../02-ai-ml-tooling/diffusers.ipynb).

## 1. What & Why

A video model denoises a whole **clip** at once — a tensor of shape
`(frames, channels, height, width)` — rather than a frame at a time. That is the crucial
architectural commitment: the model sees all frames simultaneously, so it can enforce
consistency between them, and it pays for the privilege in memory and compute.

The naive alternative, generating frames independently and stitching them, fails
immediately and instructively: each frame is a valid sample from the image distribution,
but consecutive samples are unrelated, so the result flickers violently. Even
conditioning each frame on the previous one (autoregressive) accumulates drift — small
errors compound until the subject has quietly become a different subject.

**Reach for video diffusion when** you need generated motion: text-to-video,
image-to-video (animate a still), video inpainting, or frame interpolation with genuinely
new content rather than blended pixels.

**Don't when** the task is really per-frame image processing — style transfer,
upscaling, segmentation — where a per-frame model plus explicit temporal smoothing is
cheaper and more controllable. And be realistic about cost: a few seconds of video is
minutes of GPU time on a large card.

## 2. Mental Model

**A flip-book that must be drawn all at once.**

An image model draws one picture. A video model draws every page of a flip-book
simultaneously, with the constraint that adjacent pages must depict the same world one
instant apart. It cannot draw page 1, then page 2 — by page 40 it would have forgotten
what the character looked like. So it holds every page in view and denoises them
together, and *that* is why the memory cost is what it is.

The second half of the model is where the coherence actually comes from. Think of two
kinds of conversation happening inside each layer:

- **Spatial attention** — pixels within one frame talking to each other, exactly as in
  an image model. This makes each frame look right.
- **Temporal attention** — the *same* pixel position talking to itself across frames.
  This makes the frames agree.

Almost every video architecture is a scheme for interleaving those two cheaply, because
letting every pixel talk to every pixel in every frame — full 3-D attention — is
unaffordable at any useful resolution. Example 1 puts numbers on "unaffordable".

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Latent video diffusion** | Run diffusion in a VAE latent space, as with images. Compression is usually spatial *and* temporal (e.g. 8×8×4). |
| **Temporal VAE** | A VAE that compresses across time too, so 4 input frames become 1 latent frame. Multiplies the saving; risks temporal blur. |
| **Inflation** | Take a pretrained *image* model and insert temporal layers, initialised so the model starts out reproducing image behaviour. How most video models are bootstrapped. |
| **Factorised (spatio-temporal) attention** | Spatial attention within frames, then temporal attention across frames. Cheap approximation to full 3-D attention. |
| **Full 3-D attention** | Every token attends to every token in the clip. Best quality, quadratic in total tokens — what modern DiT-based video models spend their budget on. |
| **3-D U-Net** | The earlier design: a U-Net with temporal convolutions/attention added. Largely superseded by transformers. |
| **Video DiT** | Transformer over spacetime patches (the Sora-class design). Scales better than U-Nets. |
| **Image-to-video (I2V)** | Condition on a first frame. Far easier to control than pure text-to-video and the most practical mode today. |
| **Cascade / chunked generation** | Generate keyframes, then interpolate; or generate overlapping windows and blend. How models exceed their trained clip length. |
| **Motion conditioning** | Explicit control of motion magnitude (a "motion bucket" score) or camera path. |
| **Temporal flicker** | The signature artefact: high-frequency variation between frames that should be static. |
| **Drift** | Slow accumulation of change over a long clip — the subject's identity or the lighting migrating. |

## 4. Setup

NumPy only. The two things worth computing here are the cost arithmetic that dictates
every architecture choice, and a coherence metric that shows why per-frame quality is not
enough. Real generation is a large model download and is shown as API shape in Example 4.

In [1]:
# %pip install numpy
# For real generation (Example 4): %pip install torch diffusers transformers accelerate

import os
import numpy as np

rng = np.random.default_rng(0)
RUN_HEAVY = os.getenv("PRAXIS_RUN_HEAVY") == "1"
print("numpy", np.__version__)
print("PRAXIS_RUN_HEAVY =", RUN_HEAVY)

numpy 2.5.1
PRAXIS_RUN_HEAVY = False


## 5. Worked Examples

### Example 1 — why nobody uses full 3-D attention at pixel resolution

Attention is quadratic in sequence length, and a video's sequence length is
`frames × H × W`. This is the calculation that forces every other design decision.

In [2]:
def attention_cost(frames, h, w, mode):
    '''Relative attention FLOPs (proportional to the number of query-key pairs).'''
    tokens_per_frame = h * w
    total = frames * tokens_per_frame
    if mode == "full_3d":
        return total ** 2
    if mode == "spatial_only":                 # each frame independent
        return frames * tokens_per_frame ** 2
    if mode == "factorised":                   # spatial within frame + temporal across
        return frames * tokens_per_frame ** 2 + tokens_per_frame * frames ** 2
    raise ValueError(mode)

print("Pixel space, 512x512 -- 32x32 tokens after an 8x patch/VAE downsample:")
h = w = 32
print(f"\n{'frames':>7} {'tokens':>9} {'full 3-D':>14} {'factorised':>14} {'ratio':>8}")
for frames in (1, 8, 16, 32, 64, 128):
    full = attention_cost(frames, h, w, "full_3d")
    fact = attention_cost(frames, h, w, "factorised")
    print(f"{frames:7d} {frames*h*w:9d} {full:14.3e} {fact:14.3e} {full/fact:7.1f}x")

print("\nA 128-frame clip (about 5 seconds) costs ~120x more in full 3-D attention than")
print("factorised. That factor is the entire reason temporal layers exist.")

print("\nAnd the scaling within full 3-D attention alone:")
base = attention_cost(8, h, w, "full_3d")
for frames in (8, 16, 32, 64):
    print(f"  {frames:3d} frames: {attention_cost(frames, h, w, 'full_3d')/base:6.1f}x the cost of 8 frames")
print("  -> doubling the duration QUADRUPLES the cost. Linear in seconds it is not.")

Pixel space, 512x512 -- 32x32 tokens after an 8x patch/VAE downsample:

 frames    tokens       full 3-D     factorised    ratio
      1      1024      1.049e+06      1.050e+06     1.0x
      8      8192      6.711e+07      8.454e+06     7.9x
     16     16384      2.684e+08      1.704e+07    15.8x
     32     32768      1.074e+09      3.460e+07    31.0x
     64     65536      4.295e+09      7.130e+07    60.2x
    128    131072      1.718e+10      1.510e+08   113.8x

A 128-frame clip (about 5 seconds) costs ~120x more in full 3-D attention than
factorised. That factor is the entire reason temporal layers exist.

And the scaling within full 3-D attention alone:
    8 frames:    1.0x the cost of 8 frames
   16 frames:    4.0x the cost of 8 frames
   32 frames:   16.0x the cost of 8 frames
   64 frames:   64.0x the cost of 8 frames
  -> doubling the duration QUADRUPLES the cost. Linear in seconds it is not.


Two things follow directly, and they explain most of what you see in the field:

- **Everything happens in latent space**, with aggressive spatial *and temporal*
  compression. A temporal VAE that turns 4 frames into 1 latent frame cuts the token
  count 4× and the full-attention cost 16×.
- **Clips are short.** Most open models generate 2–6 seconds natively; longer output is
  built by chaining or interpolating, not by generating it in one pass.

### Example 2 — per-frame quality does not measure coherence

Three synthetic "videos" of a moving bright spot. Every frame in all three is an equally
plausible single image. Only one of them is a usable video.

In [3]:
F, H, W = 16, 24, 24

def render(centres, sigma=2.5):
    '''Draw a Gaussian blob at the given centre in each frame.'''
    yy, xx = np.mgrid[0:H, 0:W]
    return np.stack([np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * sigma**2))
                     for cy, cx in centres])

t = np.arange(F)
coherent = render([(12 + 6*np.sin(i/3), 4 + i) for i in t])            # smooth motion
flicker = render([(12 + 6*np.sin(i/3) + rng.normal(0, 3),              # jitters
                   4 + i + rng.normal(0, 3)) for i in t])
teleport = render([(rng.uniform(4, 20), rng.uniform(4, 20)) for _ in t])  # independent frames

def per_frame_quality(v):
    '''A stand-in for an image metric: every frame is a clean, well-formed blob.'''
    return float(np.mean([f.max() for f in v]))

def temporal_smoothness(v):
    '''Mean absolute difference between consecutive frames -- lower is smoother.'''
    return float(np.abs(np.diff(v, axis=0)).mean())

def motion_consistency(v):
    '''Second difference of the tracked centroid: is the motion smooth, or erratic?'''
    yy, xx = np.mgrid[0:H, 0:W]
    cy = np.array([(f*yy).sum()/f.sum() for f in v])
    cx = np.array([(f*xx).sum()/f.sum() for f in v])
    return float(np.mean(np.abs(np.diff(cy, 2))) + np.mean(np.abs(np.diff(cx, 2))))

print(f"{'video':12} {'per-frame quality':>18} {'frame delta':>13} {'motion jerk':>13}")
for name, v in [("coherent", coherent), ("flicker", flicker), ("teleport", teleport)]:
    print(f"{name:12} {per_frame_quality(v):18.4f} {temporal_smoothness(v):13.4f} "
          f"{motion_consistency(v):13.4f}")

print("\nThe per-frame column agrees to within 1% -- every frame in all three clips is")
print("an equally well-formed blob, so an image-quality metric cannot separate them.")
print("The temporal columns differ by a factor of ~20 and 40.")
print("\nThis is why video models are not evaluated with image metrics, and why a model")
print("that scores well on FID per frame can still produce unwatchable video.")

video         per-frame quality   frame delta   motion jerk
coherent                 0.9950        0.0344        0.4574
flicker                  0.9898        0.0772        8.9674
teleport                 0.9861        0.1046       17.4091

The per-frame column agrees to within 1% -- every frame in all three clips is
an equally well-formed blob, so an image-quality metric cannot separate them.
The temporal columns differ by a factor of ~20 and 40.

This is why video models are not evaluated with image metrics, and why a model
that scores well on FID per frame can still produce unwatchable video.


### Example 3 — how temporal attention actually removes flicker

Temporal attention lets each spatial position average over its own history. Applying it
to the flickering clip above shows both the fix and its cost — the same mechanism that
removes jitter also removes genuine fast motion.

In [4]:
def temporal_attention(video, window=3.0):
    '''Attention across the time axis, per spatial position.

    Q/K are the frame indices (so attention is a learned-width temporal blur here) and
    V is the pixel value. A real model computes Q/K from content, but the shape of the
    operation -- and its effect -- is this.
    '''
    F = video.shape[0]
    idx = np.arange(F)
    scores = -((idx[:, None] - idx[None, :]) ** 2) / (2 * window ** 2)
    attn = np.exp(scores - scores.max(axis=1, keepdims=True))
    attn /= attn.sum(axis=1, keepdims=True)
    return np.einsum("ft,thw->fhw", attn, video)

print(f"{'window':>7} {'flicker: frame delta':>21} {'motion jerk':>13} | "
      f"{'coherent: jerk':>15}")
print(f"{'(none)':>7} {temporal_smoothness(flicker):21.4f} "
      f"{motion_consistency(flicker):13.4f} | {motion_consistency(coherent):15.4f}")
for wnd in (1.0, 2.0, 4.0, 8.0):
    sm_f = temporal_attention(flicker, wnd)
    sm_c = temporal_attention(coherent, wnd)
    print(f"{wnd:7.1f} {temporal_smoothness(sm_f):21.4f} {motion_consistency(sm_f):13.4f} | "
          f"{motion_consistency(sm_c):15.4f}")

print("\nA wider temporal window suppresses more flicker -- and progressively destroys")
print("the real motion in the coherent clip too. Temporal attention cannot tell the")
print("difference between 'jitter I should remove' and 'movement I should keep'")
print("unless its weights are CONTENT-dependent, which is exactly what training buys.")
print("\nIt is also the origin of the classic failure mode: over-smoothed video where")
print("everything drifts gently and nothing ever moves quickly.")

 window  flicker: frame delta   motion jerk |  coherent: jerk
 (none)                0.0772        8.9674 |          0.4574
    1.0                0.0278        1.4500 |          0.4479
    2.0                0.0161        0.3528 |          0.3328
    4.0                0.0087        0.1251 |          0.1383
    8.0                0.0034        0.0108 |          0.0127

A wider temporal window suppresses more flicker -- and progressively destroys
the real motion in the coherent clip too. Temporal attention cannot tell the
difference between 'jitter I should remove' and 'movement I should keep'
unless its weights are CONTENT-dependent, which is exactly what training buys.

It is also the origin of the classic failure mode: over-smoothed video where
everything drifts gently and nothing ever moves quickly.


### Example 4 — running a real video model

Gated behind `PRAXIS_RUN_HEAVY=1` — these are multi-gigabyte downloads and want a
substantial GPU. Image-to-video is shown first because it is by far the most controllable
mode.

In [5]:
VIDEO_CODE = '''
# --- Image-to-video: Stable Video Diffusion -------------------------------
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16, variant="fp16",
).to("cuda")
pipe.enable_model_cpu_offload()          # these models are large; usually necessary

image = load_image("start.png").resize((1024, 576))   # SVD is resolution-sensitive
frames = pipe(
    image,
    num_frames=25,
    decode_chunk_size=8,                 # decode the VAE in chunks or you will OOM
    motion_bucket_id=127,                # explicit motion magnitude: higher = more
    noise_aug_strength=0.02,             # higher = looser adherence to the input frame
    generator=torch.Generator("cuda").manual_seed(42),
).frames[0]
export_to_video(frames, "out.mp4", fps=7)

# --- Text-to-video: a DiT-based model --------------------------------------
from diffusers import CogVideoXPipeline

pipe = CogVideoXPipeline.from_pretrained(
    "THUDM/CogVideoX-2b", torch_dtype=torch.float16).to("cuda")
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()                 # temporal VAE decode is the memory spike

video = pipe(
    prompt="a paper boat drifting down a rain-filled gutter, autumn leaves",
    num_frames=49, num_inference_steps=50, guidance_scale=6.0,
    generator=torch.Generator("cuda").manual_seed(0),
).frames[0]
export_to_video(video, "out.mp4", fps=8)
'''

if RUN_HEAVY:
    import torch
    from diffusers import StableVideoDiffusionPipeline
    from diffusers.utils import load_image, export_to_video
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipe = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
    pipe.enable_model_cpu_offload()
    img = load_image("start.png").resize((1024, 576))
    frames = pipe(img, num_frames=25, decode_chunk_size=8,
                  generator=torch.Generator(device).manual_seed(42)).frames[0]
    export_to_video(frames, "out.mp4", fps=7)
    print(f"wrote out.mp4 ({len(frames)} frames)")
else:
    print("PRAXIS_RUN_HEAVY is not set - showing the API without downloading weights:")
    print(VIDEO_CODE)

PRAXIS_RUN_HEAVY is not set - showing the API without downloading weights:

# --- Image-to-video: Stable Video Diffusion -------------------------------
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16, variant="fp16",
).to("cuda")
pipe.enable_model_cpu_offload()          # these models are large; usually necessary

image = load_image("start.png").resize((1024, 576))   # SVD is resolution-sensitive
frames = pipe(
    image,
    num_frames=25,
    decode_chunk_size=8,                 # decode the VAE in chunks or you will OOM
    motion_bucket_id=127,                # explicit motion magnitude: higher = more
    noise_aug_strength=0.02,             # higher = looser adherence to the input frame
    generator=torch.Generator("cuda").manual_seed(42),
).frames[0]
export_to_video(frames

## 6. Gotchas & Pitfalls

- **Assuming duration scales linearly.** Example 1: attention is quadratic in total
  tokens, so twice the frames is roughly four times the cost. Budget accordingly.
- **VAE decode is the memory spike, not the denoising.** Decoding all frames at once
  routinely out-of-memories a card that handled the diffusion fine. `decode_chunk_size`
  and VAE tiling exist for this and should be your first move.
- **Evaluating with image metrics.** Example 2. Per-frame FID says nothing about
  flicker. Use temporal metrics, and watch the video — human review is still the honest
  benchmark here.
- **Expecting arbitrary length.** Models are trained at a fixed frame count. Asking for
  more usually degrades badly rather than erroring. Long video is chained or
  interpolated, and the seams are a real engineering problem.
- **Text-to-video when you meant image-to-video.** I2V is dramatically more controllable:
  you fix the subject, composition and style in a still image (where you have every
  image-model tool available), and the video model only has to supply motion.
- **Over-smoothing.** Example 3. Too much temporal consistency produces video where
  nothing moves convincingly. If output looks like a slowly drifting photograph, motion
  conditioning is usually the knob.
- **Ignoring frame rate at export.** Models are trained at a specific fps. Exporting 25
  frames at 24 fps when the model assumed 7 gives you a video in fast-forward.
- **Physics is not learned, it is imitated.** These models have no dynamics engine.
  Object permanence, collisions and counts fail in characteristic ways, and no prompt
  fixes it.
- **Trusting a cherry-picked demo reel.** Failure rates for text-to-video remain high;
  published examples are heavily selected. Budget for generating many candidates.

## 7. When to Use vs Alternatives

| Need | Reach for |
|---|---|
| Animate an existing still | **Image-to-video** (SVD, and I2V modes of the DiT models) — the most controllable option |
| Generate a clip from a prompt | **Text-to-video** DiT models (CogVideoX, HunyuanVideo, Wan) |
| Add motion to an image-model style/character | **AnimateDiff** — a motion module dropped onto an existing SD checkpoint plus its LoRAs |
| Smooth slow motion between real frames | Classical frame interpolation (RIFE, optical flow) — far cheaper and more faithful |
| Per-frame editing of real footage | An image model plus explicit temporal constraints, or a video-to-video pipeline |
| Understanding the backbone | [Diffusion Transformers](diffusion-transformer.ipynb) and [Attention Mechanisms](attention-mechanisms.ipynb) |

**The honest state of play.** Video generation is markedly less reliable than image
generation, and the gap is mostly Example 1: the compute wall means less capacity per
frame, fewer training tokens per unit of content, and short clips. Image-to-video is
where the practical value currently sits, because it removes most of the burden — the
model supplies motion only.

Architecturally the field has converged on **transformers over spacetime patches** with
3-D attention in a heavily-compressed latent space, having largely moved on from
inflated 3-D U-Nets. The trade in Example 1 has not gone away; it has been paid for with
better compression and more compute.

## 8. Resources

- [Video Diffusion Models](https://arxiv.org/abs/2204.03458) — Ho et al., 2022; the original, and the source of the factorised space-time attention design.
- [Align your Latents: High-Resolution Video Synthesis with Latent Diffusion Models](https://arxiv.org/abs/2304.08818) — inflating a pretrained image model into a video model.
- [Stable Video Diffusion](https://arxiv.org/abs/2311.15127) — the data-curation and training-stage argument, plus the motion-bucket conditioning used in Example 4.
- [Scalable Diffusion Models with Transformers](https://arxiv.org/abs/2212.09748) — DiT; the backbone the current generation of video models is built on.
- [AnimateDiff](https://arxiv.org/abs/2307.04725) — a plug-in motion module for existing image checkpoints.
- [CogVideoX](https://arxiv.org/abs/2408.06072) — an open text-to-video DiT with a 3-D causal VAE, described in enough detail to reimplement.
- [Diffusers: text-to-video and image-to-video](https://huggingface.co/docs/diffusers/using-diffusers/text-img2vid) — the practical API reference for everything in Example 4.